# Implementation of the example in Appendix C of the thesis of Antoine Dumas:

## Application appui plan et deux goupilles

In [1]:
import math
import sympy as sp
import numpy as np
from typing import List, Tuple

import otaf

In [2]:
def build_assembly_model(n_points_discretization=8):
    """
    Builds the symbolic compatibility and interface equations for the 3D tolerance 
    analysis mechanism (base, two pins, cover) as described in the thesis appendix C.
    
    Parameters
    ----------
    n_points_discretization : int
        Number of points used to linearize the quadratic cylindrical constraints.
        Defaults to 8, which replaces each circular zone with an octagon.
        
    Returns
    -------
    compatibility_eqs : list of sympy.Expr
        List of expressions representing the equality constraints (Cc = 0).
    interface_eqs : list of sympy.Expr
        List of expressions representing the inequality constraints (Ci <= 0).
    gap_symbols : list of sympy.Symbol
        List of the gap variables (jeux).
    deviation_symbols : list of sympy.Symbol
        List of the deviation variables (ecarts/defects) and dimension parameters.
    """
    
    # 1. Define geometrical parameters (lengths l1 to l11)
    l1, l2, l3, l4, l5, l6, l7, l8, l9, l10, l11 = sp.symbols('l1 l2 l3 l4 l5 l6 l7 l8 l9 l10 l11')
    
    # 2. Define Gap Variables (Jeux - vector 'g')
    # According to the torseurs G1a/2a, G3b/1b, G4c/1c, and G2g/1g
    u_1a2a, v_1a2a, gamma_1a2a = sp.symbols('u_1a2a v_1a2a gamma_1a2a')
    alpha_3b1b, beta_3b1b, gamma_3b1b, u_3b1b, v_3b1b, w_3b1b = sp.symbols('alpha_3b1b beta_3b1b gamma_3b1b u_3b1b v_3b1b w_3b1b')
    alpha_4c1c, beta_4c1c, gamma_4c1c, u_4c1c, v_4c1c, w_4c1c = sp.symbols('alpha_4c1c beta_4c1c gamma_4c1c u_4c1c v_4c1c w_4c1c')
    u_2g1g, v_2g1g = sp.symbols('u_2g1g v_2g1g')
    
    gap_symbols = [
        u_1a2a, v_1a2a, gamma_1a2a,
        alpha_3b1b, beta_3b1b, gamma_3b1b, u_3b1b, v_3b1b, w_3b1b,
        alpha_4c1c, beta_4c1c, gamma_4c1c, u_4c1c, v_4c1c, w_4c1c,
        u_2g1g, v_2g1g
    ]

    # 3. Define Deviation Variables (Ecarts - vector 'x')
    # Extracted from the required compatibility equations.
    alpha_1b1, alpha_2b2, alpha_2a2, alpha_1a1, alpha_2c2, alpha_1c1 = sp.symbols('alpha_1b1 alpha_2b2 alpha_2a2 alpha_1a1 alpha_2c2 alpha_1c1')
    beta_1b1, beta_2b2, beta_2a2, beta_1a1, beta_2c2, beta_1c1 = sp.symbols('beta_1b1 beta_2b2 beta_2a2 beta_1a1 beta_2c2 beta_1c1')
    w_2a2, w_1a1 = sp.symbols('w_2a2 w_1a1')
    u_1b1, u_2b2, u_2c2, u_1c1 = sp.symbols('u_1b1 u_2b2 u_2c2 u_1c1')
    v_1b1, v_2b2, v_2c2, v_1c1 = sp.symbols('v_1b1 v_2b2 v_2c2 v_1c1')
    u_2g2, u_1g1, v_2g2, v_1g1 = sp.symbols('u_2g2 u_1g1 v_2g2 v_1g1')
    
    # Diameters for the interface radii (these are combinations of nominal sizes and defects)
    d1b, d3b, d1c, d4c = sp.symbols('d1b d3b d1c d4c')
    
    deviation_symbols = [
        alpha_1b1, alpha_2b2, alpha_2a2, alpha_1a1, alpha_2c2, alpha_1c1,
        beta_1b1, beta_2b2, beta_2a2, beta_1a1, beta_2c2, beta_1c1,
        w_2a2, w_1a1,
        u_1b1, u_2b2, u_2c2, u_1c1,
        v_1b1, v_2b2, v_2c2, v_1c1,
        u_2g2, u_1g1, v_2g2, v_1g1,
        d1b, d3b, d1c, d4c,
        l1, l2, l3, l4, l5, l6, l7, l8, l9, l10, l11
    ]

    # 4. Compatibility Equations (Cc) -> 14 topological loop equations evaluated to 0
    cc = []
    
    # Boucle (1)/(2)/(3)
    cc.append(-alpha_1b1 - alpha_3b1b + alpha_2b2 - alpha_2a2 + alpha_1a1)
    cc.append(-beta_1b1 - beta_3b1b + beta_2b2 - beta_2a2 + beta_1a1)
    cc.append(-gamma_3b1b - gamma_1a2a)
    cc.append(-u_1b1 - u_3b1b + u_2b2 - u_1a2a)
    cc.append(-v_1b1 - v_3b1b + v_2b2 - v_1a2a)
    cc.append(-w_3b1b - w_2a2 + w_1a1)
    
    # Boucle (1)/(3)/(2)/(4)
    cc.append(-alpha_1b1 - alpha_3b1b + alpha_2b2 - alpha_2c2 + alpha_4c1c + alpha_1c1)
    cc.append(-beta_1b1 - beta_3b1b + beta_2b2 - beta_2c2 + beta_4c1c + beta_1c1)
    cc.append(-gamma_3b1b + gamma_4c1c)
    cc.append(-u_1b1 - u_3b1b + u_2b2 - u_2c2 + u_4c1c + l2*gamma_4c1c + u_1c1)
    cc.append(-v_1b1 - v_3b1b + v_2b2 - v_2c2 + v_4c1c - l1*gamma_4c1c + v_1c1)
    cc.append(-w_3b1b - l1*beta_2c2 + l2*alpha_2c2 + w_4c1c + l1*beta_4c1c - l2*alpha_4c1c + l1*beta_1c1 - l2*alpha_1c1)
    
    # Boucle Fonctionnelle (1)/(3)/(2)/Cf
    cc.append(-u_1b1 - l9*beta_1b1 - u_3b1b + l8*gamma_3b1b - l9*beta_3b1b + u_2b2 + l9*beta_2b2 - u_2g2 + u_2g1g + u_1g1)
    cc.append(-v_1b1 + l9*alpha_1b1 - v_3b1b + l9*alpha_3b1b - l7*gamma_3b1b + v_2b2 - l9*alpha_2b2 - v_2g2 + v_2g1g + v_1g1)

    # 5. Interface Equations (Ci) -> Linearized non-interpenetration constraints
    # The thesis defines the radius gap as: R = (D_hole - D_pin) / 2
    ci = []
    r_3b = (d1b - d3b) / 2
    r_4c = (d1c - d4c) / 2

    # Linearizing the quadratic constraint X^2 + Y^2 <= R^2 
    # into X*cos(theta) + Y*sin(theta) - R <= 0
    for i in range(n_points_discretization):
        theta = 2 * math.pi * i / n_points_discretization
        cos_t = math.cos(theta)
        sin_t = math.sin(theta)
        
        # Ci(1): 1b/3b at point A
        ci.append(u_3b1b * cos_t + v_3b1b * sin_t - r_3b)
        
        # Ci(2): 1b/3b at point B
        ci.append((u_3b1b + l3*beta_3b1b) * cos_t + (v_3b1b - l3*alpha_3b1b) * sin_t - r_3b)
        
        # Ci(3): 1c/4c at point C
        ci.append(u_4c1c * cos_t + v_4c1c * sin_t - r_4c)
        
        # Ci(4): 1c/4c at point D
        ci.append((u_4c1c + l4*beta_4c1c) * cos_t + (v_4c1c - l4*alpha_4c1c) * sin_t - r_4c)

    return cc, ci, gap_symbols, deviation_symbols

def substitute_nominal_lengths(equations, values_dict):
    """
    Helper function to swap the symbolic lengths (l1..l11) for their numerical values 
    prior to feeding them into the SystemOfConstraintsAssemblyModel matrix parser.
    """
    return [eq.subs(values_dict) for eq in equations]

In [3]:
def apply_new_naming_convention(compatibility_eqs, interface_eqs, gap_symbols, deviation_symbols):
    """
    Updates the naming convention of the symbolic equations to an ID-based system.
    
    Old convention: {dof}_{feature_string} (e.g., u_1b1, alpha_3b1b)
    New convention: {dof}_d_{ID} for deviations, {dof}_g_{ID} for gaps.
    
    Parameters
    ----------
    compatibility_eqs : list of sympy.Expr
    interface_eqs : list of sympy.Expr
    gap_symbols : list of sympy.Symbol
    deviation_symbols : list of sympy.Symbol
    
    Returns
    -------
    new_comp_eqs : list of sympy.Expr
    new_int_eqs : list of sympy.Expr
    new_gap_symbols : list of sympy.Symbol
    new_deviation_symbols : list of sympy.Symbol
    dev_id_map : dict (for reference/debugging)
    gap_id_map : dict (for reference/debugging)
    """
    
    # 1. Dynamically extract unique feature suffixes from the old symbols
    dev_features = []
    for sym in deviation_symbols:
        if '_' in sym.name:
            suffix = sym.name.split('_', 1)[1]
            if suffix not in dev_features:
                dev_features.append(suffix)
                
    gap_features = []
    for sym in gap_symbols:
        if '_' in sym.name:
            suffix = sym.name.split('_', 1)[1]
            if suffix not in gap_features:
                gap_features.append(suffix)

    # 2. Assign consecutive integer IDs to each unique feature
    dev_id_map = {feat: i+1 for i, feat in enumerate(dev_features)}
    gap_id_map = {feat: i+1 for i, feat in enumerate(gap_features)}
    
    subs_dict = {}
    new_gap_symbols = []
    new_deviation_symbols = []
    
    # 3. Remap Gap Symbols (Jeux)
    for sym in gap_symbols:
        if '_' in sym.name:
            dof, feat = sym.name.split('_', 1)
            new_name = f"{dof}_g_{gap_id_map[feat]}"
            new_sym = sp.Symbol(new_name)
            subs_dict[sym] = new_sym
            new_gap_symbols.append(new_sym)
        else:
            new_gap_symbols.append(sym)
            
    # 4. Remap Deviation Symbols (Ecarts) b
    for sym in deviation_symbols:
        if '_' in sym.name:
            dof, feat = sym.name.split('_', 1)
            new_name = f"{dof}_d_{dev_id_map[feat]}"
            new_sym = sp.Symbol(new_name)
            subs_dict[sym] = new_sym
            new_deviation_symbols.append(new_sym)
        else:
            new_deviation_symbols.append(sym)
            
    # 5. Substitute the symbols globally in the equations
    new_comp_eqs = [eq.subs(subs_dict) for eq in compatibility_eqs]
    new_int_eqs = [eq.subs(subs_dict) for eq in interface_eqs]
    
    return new_comp_eqs, new_int_eqs, new_gap_symbols, new_deviation_symbols, dev_id_map, gap_id_map

In [11]:
# Generate the symbolic equations
n_points = 8
compatibility_eqs, interface_eqs, gaps, devs = build_assembly_model(n_points_discretization=n_points)

# Table C.1 nominal dimensions
nominal_lengths = {
    sp.Symbol('l1'): 100, sp.Symbol('l2'): 40,  sp.Symbol('l3'): 30,
    sp.Symbol('l4'): 30,  sp.Symbol('l5'): 20,  sp.Symbol('l6'): 20,
    sp.Symbol('l7'): 120, sp.Symbol('l8'): 50,  sp.Symbol('l9'): 40,
    sp.Symbol('l10'): 50, sp.Symbol('l11'): -30
}

# Substitute length values
compatibility_eqs_num = substitute_nominal_lengths(compatibility_eqs, nominal_lengths)
interface_eqs_num = substitute_nominal_lengths(interface_eqs, nominal_lengths)

print(f"Generated {len(compatibility_eqs_num)} compatibility equations.")
print(f"Generated {len(interface_eqs_num)} interface equations (linearized from 4 quadratic constraints with {n_points} points).")

Generated 14 compatibility equations.
Generated 32 interface equations (linearized from 4 quadratic constraints with 8 points).


In [12]:
compatibility_eqs

[alpha_1a1 - alpha_1b1 - alpha_2a2 + alpha_2b2 - alpha_3b1b,
 beta_1a1 - beta_1b1 - beta_2a2 + beta_2b2 - beta_3b1b,
 -gamma_1a2a - gamma_3b1b,
 -u_1a2a - u_1b1 + u_2b2 - u_3b1b,
 -v_1a2a - v_1b1 + v_2b2 - v_3b1b,
 w_1a1 - w_2a2 - w_3b1b,
 -alpha_1b1 + alpha_1c1 + alpha_2b2 - alpha_2c2 - alpha_3b1b + alpha_4c1c,
 -beta_1b1 + beta_1c1 + beta_2b2 - beta_2c2 - beta_3b1b + beta_4c1c,
 -gamma_3b1b + gamma_4c1c,
 gamma_4c1c*l2 - u_1b1 + u_1c1 + u_2b2 - u_2c2 - u_3b1b + u_4c1c,
 -gamma_4c1c*l1 - v_1b1 + v_1c1 + v_2b2 - v_2c2 - v_3b1b + v_4c1c,
 -alpha_1c1*l2 + alpha_2c2*l2 - alpha_4c1c*l2 + beta_1c1*l1 - beta_2c2*l1 + beta_4c1c*l1 - w_3b1b + w_4c1c,
 -beta_1b1*l9 + beta_2b2*l9 - beta_3b1b*l9 + gamma_3b1b*l8 - u_1b1 + u_1g1 + u_2b2 + u_2g1g - u_2g2 - u_3b1b,
 alpha_1b1*l9 - alpha_2b2*l9 + alpha_3b1b*l9 - gamma_3b1b*l7 - v_1b1 + v_1g1 + v_2b2 + v_2g1g - v_2g2 - v_3b1b]

In [5]:
# Changing the naming convention to be inline with the convention of our thesis
(
    comp_eqs_renamed, 
    int_eqs_renamed, 
    gaps_renamed, 
    devs_renamed, 
    dev_map, 
    gap_map
) = apply_new_naming_convention(compatibility_eqs, interface_eqs, gaps, devs)

print("--- Mapping Reference ---")
print(f"Deviation Features mapped to IDs: {dev_map}")
print(f"Gap Features mapped to IDs: {gap_map}\n")

print("--- Symbol Verification ---")
print(f"Sample Old Gap: {gaps[3]} -> New Gap: {gaps_renamed[3]}")
print(f"Sample Old Dev: {devs[0]} -> New Dev: {devs_renamed[0]}")

--- Mapping Reference ---
Deviation Features mapped to IDs: {'1b1': 1, '2b2': 2, '2a2': 3, '1a1': 4, '2c2': 5, '1c1': 6, '2g2': 7, '1g1': 8}
Gap Features mapped to IDs: {'1a2a': 1, '3b1b': 2, '4c1c': 3, '2g1g': 4}

--- Symbol Verification ---
Sample Old Gap: alpha_3b1b -> New Gap: alpha_g_2
Sample Old Dev: alpha_1b1 -> New Dev: alpha_d_1


In [6]:
# Substitute length values
compatibility_eqs_num = substitute_nominal_lengths(comp_eqs_renamed, nominal_lengths)
interface_eqs_num = substitute_nominal_lengths(int_eqs_renamed, nominal_lengths)

print(f"Generated {len(compatibility_eqs_num)} compatibility equations.")
print(f"Generated {len(interface_eqs_num)} interface equations (linearized from 4 quadratic constraints with {n_points} points).")

Generated 14 compatibility equations.
Generated 32 interface equations (linearized from 4 quadratic constraints with 8 points).


In [7]:
model = otaf.SystemOfConstraintsAssemblyModel(compatibility_eqs_num, interface_eqs_num)
model.embedOptimizationVariable()

In [8]:
dev_map

{'1b1': 1,
 '2b2': 2,
 '2a2': 3,
 '1a1': 4,
 '2c2': 5,
 '1c1': 6,
 '2g2': 7,
 '1g1': 8}

In [9]:
model.test_zero_deviation_feasibility()

{'success': True,
 'status': 0,
 'message': 'Optimization terminated successfully. (HiGHS Status 7: Optimal)',
 'gap_values': array([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.,
        -0., -0., -0., -0., -0.]),
 'objective': 0.0}

In [10]:
# Here we have a distribution in the standard space. 
defect_distribution = otaf.distribution.get_composed_normal_defect_distribution(model.deviation_symbols)